# Reproduce HLA-Bench's published numbers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jasonbrelsford/verifiable-science-envs/blob/main/bench/reproduce.ipynb)

This notebook recomputes the headline figures published for **HLA-Bench-A** from the run
artifacts committed to the public repository, then prints each published figure next to the
recomputed one with a pass or fail.

Run it top to bottom. Runtime is about a minute.

## What this proves

That every published headline figure is the number the committed per-model result artifacts
actually contain, computed here, from the raw files, in front of you. If a published figure
and the artifacts disagree, a check below fails and says so.

## What this does not prove

* **It does not re-run any model.** The per-model result files were produced by earlier runs
  against a pinned IPD-IMGT/HLA release. They are taken as given. Nothing here can tell you
  whether the original runs were carried out correctly, only whether the published figures
  match what those runs recorded.
* **It does not re-grade any response.** The grader ran at benchmark time; the result files
  record its per-task verdicts. This notebook reads and aggregates those verdicts. It does
  print the grader's own source, from the installed package, so you can read the rule that
  produced them.
* **It does not validate the reference database.** The IPD-IMGT/HLA release the tasks were
  built from is pinned by md5 in the run manifest, which is printed below, but not refetched.

To actually re-run a model, see `hla-bench auto` in the repository README. That needs API
keys or a local Ollama, so it is deliberately out of scope here.

## What it needs

An internet connection. No API keys, no local checkout, no paid services, nothing to
download by hand.


## 1. Pin everything to one commit

Every artifact and the package itself come from commit `f8c5cbb76e5f`, not from `main`, so this
notebook keeps producing the same answer no matter what changes later.

That commit is not arbitrary: it is the one that last wrote the HLA-Bench-A run artifacts.
Pinning to it means the result files read below are the files as they were written, and the
grader source printed below is the grader as it was when those files were scored. Both have
been checked unchanged on `main` since.

If the pinned commit is ever unreachable, the next cell falls back to `main` and says so
loudly, because a fallback result is a weaker claim than a pinned one.


In [ ]:
PIN = "f8c5cbb76e5f9a9ffc08108fa1e211beb9146a64"
REPO = "jasonbrelsford/verifiable-science-envs"

import subprocess, sys, urllib.request, urllib.error


def reachable(ref):
    url = f"https://raw.githubusercontent.com/{REPO}/{ref}/runs/hla-bench-a/manifest.json"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "hla-bench-reproduce"})
        with urllib.request.urlopen(req, timeout=60):
            return True
    except Exception:
        return False


if reachable(PIN):
    REF = PIN
    PINNED = True
else:
    REF = "main"
    PINNED = False
    print("!" * 78)
    print("! The pinned commit is unreachable. Falling back to `main`.")
    print("! Everything below still runs, but it is no longer pinned: the artifacts may have")
    print("! changed since this notebook was written. Treat the result as indicative.")
    print("!" * 78)
    print()

RAW = f"https://raw.githubusercontent.com/{REPO}/{REF}"

# The package has no third-party dependencies, so this is a small, fast, pure-Python install.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", f"git+https://github.com/{REPO}@{REF}"],
    check=True,
)

import pathlib, sci_envs

print("repo:   ", REPO)
print("ref:    ", REF, "(pinned commit)" if PINNED else "(FALLBACK, not pinned)")
print("package:", pathlib.Path(sci_envs.__file__).parent)


## 2. Fetch the artifacts

The figures come from one file per model per split, written by the benchmark harness at run
time and committed:

```
runs/hla-bench-a/results/<model>.<split>.json
```

Each file holds that run's aggregate counts: overall accuracy, accuracy by tier / subtype /
slice, the fabrication counts, and a histogram of primary failure modes.
`runs/hla-bench-a/manifest.json` holds the suite's shape and the md5 of every reference file
the tasks were generated from.

The list of result files is read from the GitHub tree at the pinned commit rather than typed
out here, so no model can be quietly left out of the range. If the unauthenticated GitHub API
is rate-limited from your IP, the cell falls back to a hard-coded list and says so.


In [ ]:
import json, urllib.request, urllib.error, hashlib

RESULT_DIR = "runs/hla-bench-a/results"

# Used only if the GitHub API is rate-limited. Kept in sync by the test suite.
FALLBACK = [
    "anthropic__claude-sonnet-4-6.all.json", "baseline-cautious-abstainer.all.json",
    "baseline-cautious-abstainer.dev.json", "baseline-confident-guesser.all.json",
    "baseline-confident-guesser.dev.json", "baseline-naive-string.all.json",
    "baseline-naive-string.dev.json", "ollama__gemma3_12b.all.json",
    "ollama__llama3.1_8b.all.json", "ollama__llama3.2_3b.all.json",
    "ollama__mistral_7b.all.json", "ollama__phi4-mini.all.json",
    "ollama__qwen2.5_14b.all.json", "ollama__qwen2.5_3b.all.json",
    "ollama__qwen2.5_7b.all.json", "ollama__qwen2.5_7b.dev.json",
    "oracle-reference.all.json", "oracle-reference.dev.json",
]


def get(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "hla-bench-reproduce"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()


def list_result_files():
    url = f"https://api.github.com/repos/{REPO}/contents/{RESULT_DIR}?ref={REF}"
    try:
        entries = json.loads(get(url))
        names = sorted(e["name"] for e in entries if e["name"].endswith(".json"))
        return names, f"GitHub tree at {REF[:12]}"
    except Exception as e:
        return sorted(FALLBACK), f"hard-coded fallback ({type(e).__name__})"


names, how = list_result_files()
print(f"{len(names)} result files, listed from: {how}\n")

results, digests = {}, {}
for name in names:
    raw = get(f"{RAW}/{RESULT_DIR}/{name}")
    digests[name] = hashlib.sha256(raw).hexdigest()[:12]
    results[name] = json.loads(raw)

manifest = json.loads(get(f"{RAW}/runs/hla-bench-a/manifest.json"))

for name in names:
    d = results[name]
    print(f"  sha256:{digests[name]}  {name:42s}  {d['model']:30s} split={d['split']:4s} n={d['n']}")

print()
print("benchmark:     ", manifest["benchmark"])
print("total tasks:   ", manifest["total"])
print("split:         ", manifest["split"])
print("IPD-IMGT/HLA:  ", manifest["reference"]["release"], manifest["reference"]["tag"])
print("Allelelist md5:", manifest["reference"]["md5"]["Allelelist.txt"])


## 3. What counts as a fabricated name

Printed from the grader's own source in the package installed above, not paraphrased.

Three rules matter for the fabrication metric:

1. **Where names are looked for.** Every allele-shaped token in the model's `answer` *and* in
   its `reasoning` is classified against the pinned release. A name invented while explaining
   counts exactly as much as one invented in the answer.
2. **What "fabricated" means.** A token is `hallucinated` when it is allele-shaped but appears
   in no release back to 1.05.0. A name that merely went out of date is `deleted`, not
   fabricated, and is counted separately.
3. **What happens to an unparseable response.** If the response is not JSON with an `answer`
   key, the grader records `malformed_response` and returns **no** fabricated names. This is
   the reason the `claude-sonnet-4-6` row is a lower bound, and the checks below test for it
   explicitly.

The published metric, `Fabricated / task`, is then total fabricated tokens divided by total
tasks in the split. It is a rate per task, not a share of tasks: one task can contribute
several.


In [ ]:
import importlib, inspect

# Import the MODULES, not the re-exported `grade` function of the same name.
G = importlib.import_module("sci_envs.families.nomenclature.grade")
imgt = importlib.import_module("sci_envs.reference.imgt")


def show(title, text):
    print(f"--- {title} " + "-" * max(0, 74 - len(title)))
    print(text.rstrip())
    print()


src = inspect.getsource(G)


def block(start, end):
    lines = src.splitlines()
    i = next(k for k, ln in enumerate(lines) if start in ln)
    j = next(k for k, ln in enumerate(lines[i:], i) if end in ln)
    return "\n".join(lines[i:j + 1])


show("grade.py: an unparseable response yields NO fabricated names",
     block("r = _parse_response(response)", 'primary_failure_mode="malformed_response"'))

show("grade.py: fabrication is read from the answer AND the reasoning",
     block("# ---- hallucination", "ans_cls = ref.classify_tokens"))

show("grade.py: how the published rate is aggregated",
     block('"hallucination": {', '"top20"'))

show("imgt.py: classify_tokens, which decides valid / group / deleted / hallucinated",
     inspect.getsource(imgt.ImgtReference.classify_tokens))

print("grader version:", G.GRADER_VERSION)


## 4. Recompute

Every row below is read straight out of the artifacts fetched in step 2. Nothing is typed in.

The published range covers the **language models** on the **full 550-task suite**. The three
scripted baselines (`baseline-naive-string`, `baseline-confident-guesser`,
`baseline-cautious-abstainer`) and `oracle-reference` are reference points, not models, and
are shown separately.


In [ ]:
SCRIPTED = {"baseline-naive-string", "baseline-confident-guesser",
            "baseline-cautious-abstainer", "oracle-reference"}

rows = []
for name, d in results.items():
    ea = d["by_subtype"]["expand_ambiguity"]
    rows.append({
        "model": d["model"],
        "split": d["split"],
        "n": d["n"],
        "rate": d["hallucination"]["rate_per_task"],
        "fab_tasks": d["hallucination"]["tasks_with_hallucinated_names"],
        "acc": d["overall"]["acc"],
        "malformed": d["primary_failure_modes"].get("malformed_response", 0),
        "ea_correct": ea["correct"],
        "ea_n": ea["n"],
        "scripted": d["model"] in SCRIPTED,
    })

hdr = f"{'model':30s} {'split':5s} {'n':>4s} {'fab/task':>9s} {'fab tasks':>9s} {'acc':>7s} {'malformed':>10s} {'expand_amb':>11s}"
print(hdr)
print("-" * len(hdr))
for r in sorted(rows, key=lambda r: (r["scripted"], -r["rate"])):
    tag = "  (reference, not a model)" if r["scripted"] else ""
    print(f"{r['model']:30s} {r['split']:5s} {r['n']:4d} {r['rate']:9.4f} {r['fab_tasks']:9d} "
          f"{r['acc']:7.4f} {r['malformed']:10d} {str(r['ea_correct']) + '/' + str(r['ea_n']):>11s}{tag}")

full = [r for r in rows if r["split"] == "all" and not r["scripted"]]
lo = min(full, key=lambda r: r["rate"])
hi = max(full, key=lambda r: r["rate"])

print()
print(f"language models on the full suite: {len(full)}")
print(f"lowest  fabrication rate: {lo['rate']:.4f} -> {lo['rate']:.2f}/task  ({lo['model']})")
print(f"highest fabrication rate: {hi['rate']:.4f} -> {hi['rate']:.2f}/task  ({hi['model']})")
print(f"published range should be: {lo['rate']:.2f}-{hi['rate']:.2f} per task")


## 5. The claude-sonnet-4-6 row is a lower bound

Its responses were generated under a 600-token budget that truncated many of them. A truncated
response is not valid JSON, so the grader marks it `malformed_response` and, per the source
printed in step 3, attributes **zero** fabricated names to it, while the task still counts in
the 550-task denominator. Both the accuracy and the fabrication rate on that row are therefore
floors, not estimates.

The cell below quantifies that, and recomputes the rate over only the responses the grader
could actually read.


In [ ]:
cl = next(r for r in rows if r["model"] == "anthropic/claude-sonnet-4-6")

# rate_per_task is rounded to 4 dp in the artifact; recover the integer token count.
tokens = round(cl["rate"] * cl["n"])
assert round(tokens / cl["n"], 4) == cl["rate"], "token count is not uniquely recoverable"

parseable = cl["n"] - cl["malformed"]
print(f"tasks                                : {cl['n']}")
print(f"responses graded malformed_response  : {cl['malformed']}  ({cl['malformed'] / cl['n']:.0%})")
print(f"responses the grader could parse     : {parseable}")
print(f"fabricated tokens counted            : {tokens}")
print()
print(f"published rate, over all 550 tasks   : {cl['rate']:.4f} -> {cl['rate']:.2f}/task   (LOWER BOUND)")
print(f"rate over the {parseable} parseable responses : {tokens / parseable:.4f} -> {tokens / parseable:.2f}/task")
print()
print(f"published accuracy                   : {cl['acc']:.4f} -> {cl['acc']:.0%}          (LOWER BOUND)")


## 6. The 0.05 that used to be published was a dev-split number

`dev` is the 112-task public subset. `all` is the full 550-task suite. Quoting a `dev` figure
as the suite result understates it, which is what happened: the retired claim of
"0.05-0.14 per task" took its low end from `ollama/qwen2.5:7b` on `dev`.


In [ ]:
for r in sorted(rows, key=lambda r: (r["model"], r["split"])):
    if any(o["model"] == r["model"] and o["split"] != r["split"] for o in rows):
        print(f"{r['model']:30s} split={r['split']:4s} n={r['n']:4d} fab/task={r['rate']:.4f} -> {r['rate']:.2f}")

q_dev = next(r for r in rows if r["model"] == "ollama/qwen2.5:7b" and r["split"] == "dev")
q_all = next(r for r in rows if r["model"] == "ollama/qwen2.5:7b" and r["split"] == "all")
print()
print(f"ollama/qwen2.5:7b  dev (n={q_dev['n']}): {q_dev['rate']:.2f}/task   <- the retired 0.05")
print(f"ollama/qwen2.5:7b  all (n={q_all['n']}): {q_all['rate']:.2f}/task   <- the same model on the suite")
print()
rates = sorted({round(r["rate"], 2) for r in rows})
print("every fabrication rate that appears in any artifact, rounded:", rates)
print("is 0.14 among them?", 0.14 in rates)


## 7. Published against recomputed

Each row states a published claim, the value recomputed above, and whether they agree.
`PUBLISHED` is written out by hand on purpose: these are the strings that appear in the
repository and on the live API, and the point of the exercise is to test them.


In [ ]:
# Where each published figure appears (verifiable-science-envs unless noted):
#   fabrication range   bench/HLA-Bench-A.md, README.md, assets/llms.txt, .hf/README.md,
#                       skills/hla-verify/SKILL.md, docs/paper/hla-bench-draft.md,
#                       edge/src/mcp.js (the live `about` tool), sci_envs/mcp_server.py,
#                       and hlaverify-website src/llms.txt + public/llms.txt
#   0% expand_ambiguity  the same set, plus spaces/hla-verify/README.md
PUBLISHED = {
    "suite_n": 550,
    "dev_n": 112,
    "n_language_models": 9,
    "fab_low": 0.06,
    "fab_high": 0.20,
    "fab_low_model": "ollama/qwen2.5:3b",
    "fab_high_model": "ollama/gemma3:12b",
    "expand_ambiguity_pct": 0,
    "expand_ambiguity_n_per_model": 30,
    "claude_malformed": 187,
    "claude_rate_is_lower_bound": True,
    "retired_low_is_dev_only": 0.05,
    "retired_high_matches_nothing": 0.14,
}

checks = []


def check(claim, published, recomputed, ok=None):
    ok = (published == recomputed) if ok is None else ok
    checks.append((claim, published, recomputed, ok))


check("suite size (all split)", PUBLISHED["suite_n"], manifest["total"])
check("dev split size", PUBLISHED["dev_n"], manifest["split"]["dev"])
check("language models scored on the full suite", PUBLISHED["n_language_models"], len(full))
check("lowest fabrication rate per task", PUBLISHED["fab_low"], round(lo["rate"], 2))
check("model at the low end", PUBLISHED["fab_low_model"], lo["model"])
check("highest fabrication rate per task", PUBLISHED["fab_high"], round(hi["rate"], 2))
check("model at the high end", PUBLISHED["fab_high_model"], hi["model"])

ea = [r for r in rows if r["split"] == "all" and r["model"] != "oracle-reference"]
check("every model and baseline scores 0% on expand_ambiguity (all split)",
      PUBLISHED["expand_ambiguity_pct"], max(r["ea_correct"] for r in ea))
check("expand_ambiguity tasks per model",
      PUBLISHED["expand_ambiguity_n_per_model"], {r["ea_n"] for r in ea}.pop()
      if len({r["ea_n"] for r in ea}) == 1 else sorted({r["ea_n"] for r in ea}))
orc = next(r for r in rows if r["model"] == "oracle-reference" and r["split"] == "all")
check("oracle-reference solves expand_ambiguity, so the task is solvable",
      "30/30", f"{orc['ea_correct']}/{orc['ea_n']}")

check("claude-sonnet-4-6 responses graded malformed", PUBLISHED["claude_malformed"], cl["malformed"])
check("claude-sonnet-4-6 rate is a lower bound (malformed responses > 0 and "
      "the grader gives them no fabricated names)",
      PUBLISHED["claude_rate_is_lower_bound"], cl["malformed"] > 0)

check(f"the retired low end {PUBLISHED['retired_low_is_dev_only']} is the qwen2.5:7b DEV rate, "
      "not a suite rate",
      PUBLISHED["retired_low_is_dev_only"], round(q_dev["rate"], 2))
check(f"the retired high end {PUBLISHED['retired_high_matches_nothing']} matches no artifact",
      "absent", "absent" if PUBLISHED["retired_high_matches_nothing"] not in rates else "PRESENT")

w = max(len(c[0]) for c in checks)
print(f"{'claim':{w}s}  {'published':>22s}  {'recomputed':>22s}  result")
print("-" * (w + 60))
for claim, pub, rec, ok in checks:
    print(f"{claim:{w}s}  {str(pub):>22s}  {str(rec):>22s}  {'PASS' if ok else 'FAIL'}")

failed = [c for c in checks if not c[3]]
print()
print(f"{len(checks) - len(failed)}/{len(checks)} checks pass")
if failed:
    print("FAILED:")
    for claim, pub, rec, _ in failed:
        print(f"  {claim}: published {pub!r}, artifacts say {rec!r}")
    raise SystemExit(1)
print("ALL PASS: every published headline figure matches the committed artifacts.")


## 8. Reading the result

`ALL PASS` means the published headline figures are the figures the committed run artifacts
contain. That is the whole claim. It says nothing about whether the underlying runs were
well-designed, whether the models would score the same today, or whether the benchmark
measures something worth measuring. Those are separate arguments, and the tables, the task
specification in `docs/TASK_SPEC.md` and the grader specification in `docs/GRADER_SPEC.md`
are there to be argued with.

Two figures carry a qualifier that must travel with them:

* **`anthropic/claude-sonnet-4-6` is a lower bound** on both accuracy and fabrication rate,
  because 187 of its 550 responses were truncated and graded `malformed_response`. A clean
  re-run at a larger token budget is pending.
* **`dev` rows are a 112-task public subset**, not the suite. A `dev` figure quoted as a
  suite figure is wrong, which is how the retired "0.05" entered circulation.

If you want to go further than this notebook: clone the repository, `pip install -e ".[dev]"`,
and run `pytest -q`. The oracle test regenerates the tasks from the pinned IPD-IMGT/HLA
release and re-grades them end to end, which does exercise the generator and grader rather
than reading their recorded output.
